# OpenAI Direct Haptic Sequence Generator

Three screenshots -> OpenAI vision -> uint8 ERM RTP sequence JSON. No HapticGen, VAE, or other model is used.

In [ ]:
# Setup
import json
import os
import base64
import mimetypes
from pathlib import Path
from datetime import datetime

try:
    from google.colab import files, userdata
except ImportError:
    files = None
    userdata = None

try:
    from openai import OpenAI
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.0.0'], check=True)
    from openai import OpenAI

if not os.getenv('OPENAI_API_KEY') and userdata is not None:
    try:
        key = userdata.get('OPENAI_API_KEY')
        if key:
            os.environ['OPENAI_API_KEY'] = key
    except Exception:
        pass

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY first. In Colab, add it as a secret named OPENAI_API_KEY.')

client = OpenAI()
print('OpenAI client ready')

In [ ]:
# Parameters
MODEL = 'gpt-4.1-mini'
FLOW = 'success'  # success, error, notification, loading, generic
REPEAT = 3
SEED_LABEL = 'trial_001'
NOTES = ''

DURATION_SECONDS = 2.0
SAMPLE_INTERVAL_MS = 10
SAMPLE_COUNT = int(round(DURATION_SECONDS * 1000 / SAMPLE_INTERVAL_MS))
OUTPUT_ROOT = Path('/content/openai_direct_haptics') if Path('/content').exists() else Path('outputs/openai_direct_haptics')

print('sample_count:', SAMPLE_COUNT)
print('sample_interval_ms:', SAMPLE_INTERVAL_MS)
print('duration_seconds:', DURATION_SECONDS)

In [ ]:
# Upload or set three screenshots in chronological order: before, during, after.
# In Colab this opens an upload picker. Outside Colab, set IMAGE_PATHS manually.
IMAGE_PATHS = []

if files is not None:
    uploaded = files.upload()
    IMAGE_PATHS = [str(Path(name).resolve()) for name in uploaded.keys()]

if len(IMAGE_PATHS) != 3:
    raise RuntimeError('Provide exactly 3 screenshots in IMAGE_PATHS, ordered as before, during, after.')

print('Using screenshots:')
for path in IMAGE_PATHS:
    print(' -', path)

In [ ]:
# Helpers
SUPPORTED_FLOWS = {'success', 'error', 'notification', 'loading', 'generic'}

def image_to_data_url(path):
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or 'application/octet-stream'
    data = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{mime};base64,{data}'

def response_schema(sample_count):
    return {
        'type': 'object',
        'additionalProperties': False,
        'required': ['sequence'],
        'properties': {
            'sequence': {
                'type': 'array',
                'minItems': sample_count,
                'maxItems': sample_count,
                'items': {'type': 'integer', 'minimum': 0, 'maximum': 255},
            }
        },
    }

def build_prompt(flow, variant_index, repeat, seed_label, notes):
    return f'''
You are generating a directly usable vibrotactile sequence for an ESP32 + DRV2605L + ERM motor in RTP mode.
You receive exactly three screenshots in chronological order: before, during, after.

Return only the JSON object required by the schema.

Hard requirements:
- Generate exactly {SAMPLE_COUNT} integer values in the field sequence.
- The Arduino sends one value every {SAMPLE_INTERVAL_MS} ms.
- Total playback duration is {DURATION_SECONDS:.3f} seconds.
- Values must be uint8 integers from 0 to 255.
- Prefer 0 to 240. Avoid long runs at 255.
- The sequence must be directly usable as a DRV2605L RTP amplitude envelope.

ERM motor constraints:
- Make it very random, but physically perceivable on an ERM motor.
- Do not output pure per-sample white noise; ERM inertia will blur it.
- Use random burst lengths, random gaps, random attack times, random decays, and random peak intensities.
- Use ramps into and out of strong values so the motor can respond.
- Include silence or near-silence between some bursts.
- Keep the first and last 3 to 5 samples near 0.
- Avoid simple repeated patterns.

Flow context:
- flow = {flow}
- Use the flow only as a weak context cue for density and intensity.
- success: smoother, less aggressive random texture.
- error: denser, sharper random texture.
- notification: medium density random texture.
- loading: more continuous but irregular random texture.
- generic: balanced random texture.

Repeat context:
- seed label = {seed_label}
- variant = {variant_index} of {repeat}
- Make this variant distinct from the others while following the same constraints.

User notes:
{notes or 'None'}
'''.strip()

def validate_sequence(sequence):
    if not isinstance(sequence, list):
        raise ValueError('sequence must be a list')
    if len(sequence) != SAMPLE_COUNT:
        raise ValueError(f'expected {SAMPLE_COUNT} values, got {len(sequence)}')
    for i, value in enumerate(sequence):
        if not isinstance(value, int) or value < 0 or value > 255:
            raise ValueError(f'invalid uint8 value at index {i}: {value!r}')
    return sequence

def write_sequence(path, sequence):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(sequence, separators=(',', ':')) + '\n', encoding='utf-8')
    return path

In [ ]:
# Generate OpenAI-direct sequences
if FLOW not in SUPPORTED_FLOWS:
    raise ValueError(f'FLOW must be one of {sorted(SUPPORTED_FLOWS)}')
if len(IMAGE_PATHS) != 3:
    raise ValueError('IMAGE_PATHS must contain exactly 3 screenshots')

run_dir = OUTPUT_ROOT / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{FLOW}_{SEED_LABEL}"
run_dir.mkdir(parents=True, exist_ok=True)
image_content = [{'type': 'input_image', 'image_url': image_to_data_url(path)} for path in IMAGE_PATHS]
written = []

for variant_index in range(1, REPEAT + 1):
    prompt = build_prompt(FLOW, variant_index, REPEAT, SEED_LABEL, NOTES)
    response = client.responses.create(
        model=MODEL,
        input=[{'role': 'user', 'content': [{'type': 'input_text', 'text': prompt}] + image_content}],
        text={
            'format': {
                'type': 'json_schema',
                'name': 'direct_erm_haptic_sequence',
                'schema': response_schema(SAMPLE_COUNT),
                'strict': True,
            }
        },
        temperature=0,
    )
    payload = json.loads(response.output_text)
    sequence = validate_sequence(payload['sequence'])
    output_path = run_dir / f'{FLOW}_{variant_index:02d}.json'
    write_sequence(output_path, sequence)
    written.append(output_path)
    print('wrote', output_path)

print('done')
print('run_dir:', run_dir)

In [ ]:
# Optional: download generated JSON files in Colab
if files is not None:
    import zipfile
    zip_path = run_dir.with_suffix('.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in written:
            zf.write(path, arcname=path.name)
    files.download(str(zip_path))
    print('downloaded', zip_path)